#Question_2

In [ ]:
# 2 Solution a)

In [16]:
def decimal_to_binary(decimal): 
    """
    Convert a decimal number to its binary equivalent.
    """
    if decimal == 0:
        return "0"

    sign = "-" if decimal < 0 else ""
    decimal = abs(decimal)

    integer_part = int(decimal)
    fractional_part = decimal - integer_part
    binary = ""
    if integer_part == 0:
        binary = "0"
    else:
        while integer_part > 0:
            binary = str(integer_part % 2) + binary
            integer_part //= 2

    if fractional_part:
        binary += "."
        for _ in range(32):  # Limit digits for fractions with repeating binary expansions
            fractional_part *= 2
            bit = int(fractional_part)
            binary += str(bit)
            fractional_part -= bit
            if fractional_part == 0:
                break

    return sign + binary
n= float(input("Enter a decimal number: "))
print(n , "in binary is: ",decimal_to_binary(n))

45.45 in binary is:  101101.01110011001100110011001100110011


In [ ]:
# ques_2b


In [14]:
import math
def decimal_to_ieee(num):
    if num == 0:
        return "0" * 32  # Special case for zero

    sign_bit = '0' if num >= 0 else '1'       # represented as s
    num = abs(num)

    exponent = 0
    while num >= 2:
        num /= 2
        exponent += 1
    while num < 1:
        num *= 2
        exponent -= 1

    exponent += 1023  # Bias for double precision      #represented as e
    exponent_bits = f"{exponent:011b}"

    mantissa = ""                                          #represented as m 
    num -= 1  # Remove the leading 1 for normalized numbers
    for _ in range(52):
        num *= 2
        if num >= 1:
            mantissa += '1'
            num -= 1
        else:
            mantissa += '0'

    ieee754_binary = sign_bit + (" = s") , exponent_bits + (" = e") , mantissa + (" = m")
    return ieee754_binary
n= float(input("Enter a number to convert to IEEE double precision: "))
print(n,"in IEEE is:", decimal_to_ieee(n))


8.342349234823884e+17 in IEEE is: ('0 = s', '10000111010 = e', '0111001001111001100001111100011000101100100100101111 = m')


#qus_2c

In [6]:
def ieee_to_decimal(s, e, f):
    sign_bit = str(s)
    exponent_bits = str(e)
    fraction_bits = str(f)

    if sign_bit not in {"0", "1"}:
        raise ValueError("Sign bit must be 0 or 1")
    if not exponent_bits or any(bit not in "01" for bit in exponent_bits):
        raise ValueError("Exponent must contain only 0 and 1 bits")
    if any(bit not in "01" for bit in fraction_bits):
        raise ValueError("Fraction must contain only 0 and 1 bits")

    exponent = int(exponent_bits, 2)
    fraction = int(fraction_bits, 2) if fraction_bits else 0
    exponent_max = (1 << len(exponent_bits)) - 1
    bias = (1 << (len(exponent_bits) - 1)) - 1

    if exponent == exponent_max:
        if fraction == 0:
            return float("-inf") if sign_bit == "1" else float("inf")
        return float("nan")

    if exponent == 0:
        if fraction == 0:
            return -0.0 if sign_bit == "1" else 0.0
        value = (fraction / (2 ** len(fraction_bits))) * (2 ** (1 - bias))
    else:
        value = (1 + fraction / (2 ** len(fraction_bits))) * (
            2 ** (exponent - bias)
        )

    if sign_bit == "1":
        value = -value

    return value


s = input("Enter the sign bit (0 or 1): ")
e = input("Enter the exponent bits: ")
f = input("Enter the fraction bits: ")

print("The decimal equivalent is:", ieee_to_decimal(s, e, f))

The decimal equivalent is: 7.202517429043684e+67


In [ ]:
#2d

In [1]:
import math

def decimal_to_ieee_double(x):
    """
    Definition used in 2b:
    Converts a decimal value to IEEE-754 double-precision format:
    (s, e, m)
    s = sign bit
    e = 11-bit exponent
    m = 52-bit mantissa
    """
    # Handle special values
    if math.isnan(x):
        return ('0', '11111111111', '0000000000000000000000000000000000000000000000000001')
    if math.isinf(x):
        s = '0' if x > 0 else '1'
        return (s, '11111111111', '0000000000000000000000000000000000000000000000000000')

    if x == 0.0:
        return ('0', '00000000000', '0000000000000000000000000000000000000000000000000000')

    s = '0' if x > 0 else '1'
    y = abs(x)

    # Normalize y to [1,2)
    exponent = 0
    while y >= 2.0:
        y /= 2.0
        exponent += 1
    while y < 1.0:
        y *= 2.0
        exponent -= 1

    # Bias for double precision
    e = exponent + 1023
    exponent_bits = format(e, '011b')

    # Mantissa: remove leading 1 and keep 52 bits after decimal point
    fraction = y - 1.0
    mantissa = ""
    for _ in range(52):
        fraction *= 2.0
        if fraction >= 1.0:
            mantissa += '1'
            fraction -= 1.0
        else:
            mantissa += '0'

    return (s, exponent_bits, mantissa)


def ieee_to_decimal(s, e, m):
    """
    Definition used in 2c:
    Converts IEEE-754 double-precision bits (s, e, m) back to decimal.
    """
    s = str(s)
    e = str(e)
    m = str(m)

    if e == '11111111111':
        if m == '0' * 52:
            return float('inf') if s == '0' else float('-inf')
        return float('nan')

    exponent = int(e, 2)
    fraction = int(m, 2) / (2 ** 52)

    if exponent == 0:
        # Subnormal / zero-like case
        value = 0.0 + fraction * (2 ** (-1022))
        # For this question, normal and infinities are the main cases
    else:
        value = (1.0 + fraction) * (2 ** (exponent - 1023))

    if s == '1':
        value = -value

    return value


def verify_accuracy(values):
    for x in values:
        s, e, m = decimal_to_ieee_double(x)
        y = ieee_to_decimal(s, e, m)

        print(f"Original x = {x}")
        print(f"IEEE bits = s={s}, e={e}, m={m}")
        print(f"Recovered value = {y}")

        if math.isinf(x) or math.isinf(y):
            print("Accuracy check:", math.isinf(x) and math.isinf(y) and ((x > 0 and y > 0) or (x < 0 and y < 0)))
        elif math.isnan(x) or math.isnan(y):
            print("Accuracy check:", math.isnan(x) and math.isnan(y))
        else:
            print("Accuracy check:", x == y)
            print("Absolute error:", abs(x - y))
        print("-" * 70)

# Test values
numbers = [200.0, math.pi, float('inf'), float('-inf')]

verify_accuracy(numbers)

Original x = 200.0
IEEE bits = s=0, e=10000000110, m=1001000000000000000000000000000000000000000000000000
Recovered value = 200.0
Accuracy check: True
Absolute error: 0.0
----------------------------------------------------------------------
Original x = 3.141592653589793
IEEE bits = s=0, e=10000000000, m=1001001000011111101101010100010001000010110100011000
Recovered value = 3.141592653589793
Accuracy check: True
Absolute error: 0.0
----------------------------------------------------------------------
Original x = inf
IEEE bits = s=0, e=11111111111, m=0000000000000000000000000000000000000000000000000000
Recovered value = inf
Accuracy check: True
----------------------------------------------------------------------
Original x = -inf
IEEE bits = s=1, e=11111111111, m=0000000000000000000000000000000000000000000000000000
Recovered value = -inf
Accuracy check: True
----------------------------------------------------------------------
